In [ ]:
!pip install gradio as gr

ERROR: Could not find a version that satisfies the requirement as (from versions: none)
ERROR: No matching distribution found for as


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Load explicitly using Auto classes
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/my_final_model", local_files_only = True)
model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/my_final_model", local_files_only = True)

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

print("Model Loaded Successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model Loaded Successfully!


In [ ]:
import gradio as gr

# Store history across requests
history = []

def analyze_sentiment(review):
    if not review.strip():
        return "Please enter a review", "", history_to_display()

    result = classifier(review)[0]
    label = result["label"]
    score = result["score"]

    if label == "LABEL_1":
        sentiment = "Positive 😊"
    else:
        sentiment = "Negative 😞"

    confidence_pct = score * 100
    confidence_text = f"{confidence_pct:.1f}%"

    # Add to history
    history.append([review, sentiment, confidence_text])

    return sentiment, confidence_bar(confidence_pct), history_to_display()

def confidence_bar(pct):
    filled = int(pct / 5)       # 20 blocks total
    empty = 20 - filled
    bar = "█" * filled + "░" * empty
    return f"{bar}  {pct:.1f}%"

def history_to_display():
    if not history:
        return []
    return history[::-1]  # newest first

def clear_all():
    history.clear()
    return "", "", "", []

with gr.Blocks(title="Movie Review Sentiment Analyzer") as app:

    gr.Markdown("# 🎬 Movie Review Sentiment Analyzer")
    gr.Markdown("Type a movie review and the AI will tell you if it's **positive** or **negative**.")

    with gr.Row():
        with gr.Column():
            review_input = gr.Textbox(
                label="Your Movie Review",
                placeholder="e.g. This movie was absolutely fantastic!",
                lines=4
            )
            submit_btn = gr.Button("Analyze ✨", variant="primary")
            clear_btn = gr.Button("Clear")

        with gr.Column():
            sentiment_output = gr.Textbox(label="Sentiment")
            confidence_output = gr.Textbox(label="Confidence")  # feature 1

    # Feature 2 — Review history table
    gr.Markdown("## 📋 Review History")
    history_table = gr.Dataframe(
        headers=["Review", "Sentiment", "Confidence"],
        datatype=["str", "str", "str"],
        label="All analyzed reviews (newest first)",
        wrap=True
    )

    submit_btn.click(
        fn=analyze_sentiment,
        inputs=review_input,
        outputs=[sentiment_output, confidence_output, history_table]
    )

    clear_btn.click(
        fn=clear_all,
        inputs=None,
        outputs=[review_input, sentiment_output, confidence_output, history_table]
    )

    gr.Examples(
        examples=[
            ["This movie was absolutely fantastic! Best film I've seen this year."],
            ["Terrible movie. Waste of time and money. Awful acting."],
            ["It was okay, nothing special but not bad either."],
            ["The visuals were stunning but the plot was weak."],
        ],
        inputs=review_input
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0e0f1ba529f6ec5984.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
